# Hub-and-Spoke Multi-Agent Memory Evaluation

## The Problem

In a multi-agent system, the most important failures happen *before* the final answer is produced. A sub-agent may receive stale context, the coordinator may update the budget but not re-dispatch the hotel search, or two agents may work from different versions of the same plan. These are **memory and coordination failures** — and they're invisible unless you instrument for them.

## What This Notebook Does

We run a **travel planning** hub-and-spoke system through two sessions:

1. **Simple Session** — User requests a trip with a fixed budget. No changes mid-conversation. This is the baseline.
2. **Session with Feedback** — Same initial request, but the user changes the budget mid-session. This tests whether agents pick up the update, whether the coordinator re-dispatches correctly, and whether the final itinerary reflects the new constraints.

The metrics tell us what happened — no manual labeling of "good" or "bad" needed.

## Architecture

```


                    ┌─────────────────────┐
                    │   User / Customer    │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Travel Coordinator │  <-------- Hub (no memory)
                    │   Decides which      │    Routes queries to spokes
                    │   spoke to call      │    Compresses user message
                    └──┬───────┬────────┬──┘    into handoff query
                       │       │        │
              ┌────────▼──┐ ┌──▼─────┐ ┌▼──────────┐
              │  Flight   │ │ Hotel  │ │ Itinerary │
              │  Agent    │ │ Agent  │ │ Agent     │
              └─────┬─────┘ └───┬────┘ └─────┬─────┘
                    │           │             │
              ┌─────▼───────────▼─────────────▼─────┐
              │     AgentCore Shared Memory          │
              │     (session_id shared,              │
              │      actor_id per agent)             │
              └─────────────────────────────────────┘





```

**Flow:**
- Coordinator receives user request and delegates via tool calls
- Each spoke reads shared memory (`get_last_k_turns`), processes the query, writes back (`create_event`)
- Itinerary agent runs last — consumes Flight + Hotel outputs from memory
- All context flows through the coordinator or through shared memory

## Metrics Evaluated

**Memory Context Metrics** (LLM-as-judge, 1-5 scale):

| Metric | What it measures |
|--------|------------------|
| Context Freshness | Is the agent working with the latest information? |
| Handoff Completeness | Did the coordinator include all facts the spoke needs? |
| Context Utilization | Did the spoke use the context it read from memory? |
| State Consistency | Do all spokes agree on key facts (budget, dates, etc.)? |
| Memory Write Accuracy | Is what the agent wrote to memory factually correct? |
| Redundant Context | How much repeated/irrelevant context was transferred? |
| CCR | Length ratio of handoff vs original (pure math) |

**Memory Latency Metrics** (timers + token counts):

| Metric | What it measures |
|--------|------------------|
| Memory Read/Write Latency | Time for memory operations |
| Coordination Latency % | Fraction of time spent on coordination vs reasoning |
| Coordination Token % | Fraction of tokens spent on context vs generation |

## Step 1 — Setup

In [1]:
!pip install -qr requirements.txt

In [2]:
import os
import time
import logging
from datetime import datetime
from IPython.display import display, Markdown

from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory import MemoryClient
from botocore.exceptions import ClientError

from metrics_collector import MetricsCollector
from model_config import (
    AGENT_MODEL_ID, FLIGHT_PROMPT, HOTEL_PROMPT,
    ITINERARY_PROMPT, HUB_PROMPT,
)

region = os.getenv("AWS_REGION", "us-west-2")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("hub-spoke-metrics")

## Step 2 — Create or Reuse AgentCore Memory

The memory resource is a long-lived container. Each session uses a fresh `session_id` so agents start with a clean slate — no data leaks between runs.

In [3]:
memory_client = MemoryClient(region_name=region)
memory_name = "TravelMetrics_STM"
memory_id = None

try:
    print("Creating Memory (takes ~3 min first time)...")
    memory = memory_client.create_memory_and_wait(
        name=memory_name, description="Travel Metrics STM",
        strategies=[], max_wait=300, poll_interval=10)
    memory_id = memory["id"]
    print(f"Memory ready: {memory_id}")
except ClientError as e:
    if "already exists" in str(e):
        memories = memory_client.list_memories()
        memory_id = next(
            (m["id"] for m in memories
             if m["id"].startswith(memory_name) and m.get("status") == "ACTIVE"),
            None)
        print(f"Reusing existing memory: {memory_id}")
    else:
        raise

2026-04-20 00:11:35 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


2026-04-20 00:11:35 - INFO - Initialized MemoryClient for control plane: us-west-2, data plane: us-west-2


Creating Memory (takes ~3 min first time)...


2026-04-20 00:11:35 - ERROR - Failed to create memory: An error occurred (ValidationException) when calling the CreateMemory operation: Validation failed during CreateMemory: Memory with name TravelMetrics_STM already exists


Reusing existing memory: TravelMetrics_STM-cgOUx7FcVo


## Step 3 — Instrumented Memory Hook

Extends the standard `ShortTermMemoryHook` with:
- Latency timers around `get_last_k_turns` and `create_event`
- Metrics recording in `finally` blocks (never skipped even if memory fails)

In [4]:
class CustomInstrumentationHook(HookProvider):

    def __init__(self, mem_client, mem_id, collector, agent_name):
        self.mem_client = mem_client
        self.mem_id = mem_id
        self.collector = collector
        self.agent_name = agent_name

    def on_agent_initialized(self, event: AgentInitializedEvent):
        context = ""
        try:
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            if not actor_id or not session_id:
                return

            t0 = time.perf_counter()
            recent = self.mem_client.get_last_k_turns(
                memory_id=self.mem_id, actor_id=actor_id,
                session_id=session_id, k=5, branch_name="main")
            self.collector.record_memory_read_latency(self.agent_name, time.perf_counter() - t0)

            if recent:
                parts = []
                for turn in recent:
                    for msg in turn:
                        parts.append(f"{msg['role'].title()}: {msg['content']['text']}")
                context = "\n".join(parts)
                event.agent.system_prompt += (
                    f"\n\nRecent conversation history:\n{context}"
                    "\n\nContinue naturally based on this context.")
                logger.info(f"[{self.agent_name}] Loaded {len(recent)} turns")
        except Exception as e:
            logger.error(f"[{self.agent_name}] Memory read failed: {e}")
        finally:
            self.collector.record_retrieved_context(self.agent_name, context)

    def on_message_added(self, event: MessageAddedEvent):
        last = event.agent.messages[-1]
        if last["role"] == "assistant":
            self.collector.record_response(self.agent_name, last["content"][0]["text"])
        try:
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            if not actor_id or not session_id:
                return
            # Prefix content with agent name so other agents see who wrote it
            content = f"[{self.agent_name}] {last['content'][0]['text']}"
            t0 = time.perf_counter()
            self.mem_client.create_event(
                memory_id=self.mem_id, actor_id=actor_id,
                session_id=session_id,
                messages=[(content, last["role"])],
                metadata={"agent_name": {"stringValue": self.agent_name}})
            self.collector.record_memory_write_latency(self.agent_name, time.perf_counter() - t0)
        except Exception as e:
            logger.error(f"[{self.agent_name}] Memory write failed: {e}")

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

## Step 4 — Session Runner

Three spokes + one hub coordinator. `run_multiagent_session()` encapsulates a full session with turn-level tracking via `begin_turn()` / `end_turn()`.

In [5]:
def run_multiagent_session(conversation: list, session_label: str) -> MetricsCollector:
    """Run a full conversation session and return the metrics collector."""
    ts = datetime.now().strftime("%Y%m%d%H%M%S")
    sid = f"{session_label}-{ts}"
    # Shared actor_id so all spokes can read each other's writes
    shared_aid = f"travel-team-{ts}"

    collector = MetricsCollector(region=region)
    turn_counter = 0

    @tool
    def flight_booking_assistant(query: str) -> str:
        """Process flight booking queries.
        Args:
            query: A flight-related question
        Returns:
            Flight information and booking options
        """
        collector.record_handoff("flight", query)
        hook = CustomInstrumentationHook(memory_client, memory_id, collector, "flight")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=FLIGHT_PROMPT,
                      state={"actor_id": shared_aid, "session_id": sid})
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("flight", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("flight",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def hotel_booking_assistant(query: str) -> str:
        """Process hotel booking queries.
        Args:
            query: A hotel-related question
        Returns:
            Hotel information and booking options
        """
        collector.record_handoff("hotel", query)
        hook = CustomInstrumentationHook(memory_client, memory_id, collector, "hotel")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=HOTEL_PROMPT,
                      state={"actor_id": shared_aid, "session_id": sid})
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("hotel", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("hotel",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def itinerary_assistant(query: str) -> str:
        """Build a travel itinerary from flight and hotel results.
        Args:
            query: Request to build an itinerary
        Returns:
            A cohesive travel itinerary
        """
        collector.record_handoff("itinerary", query)
        hook = CustomInstrumentationHook(memory_client, memory_id, collector, "itinerary")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=ITINERARY_PROMPT,
                      state={"actor_id": shared_aid, "session_id": sid})
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("itinerary", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("itinerary",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    hub = Agent(system_prompt=HUB_PROMPT, model=AGENT_MODEL_ID,
               tools=[flight_booking_assistant, hotel_booking_assistant, itinerary_assistant])

    for msg in conversation:
        turn_counter += 1
        collector.begin_turn(turn_counter, msg)
        print(f"\n{'='*60}")
        print(f"[{session_label}] Turn {turn_counter}: {msg}")
        print(f"{'='*60}")
        resp = hub(msg)
        collector.end_turn()
        print(f"\nHub: {str(resp)[:300]}...")

    return collector

## Step 5 — Simple Session

Fixed budget, no mid-session changes. This is the baseline.

In [6]:
simple_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    "Now build me a day-by-day itinerary for the trip based on the flight and hotel you found.",
]

simple_conv_metrics = run_multiagent_session(simple_conversation, "simple")

2026-04-20 00:11:35 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-04-20 00:11:35 - INFO - Creating Strands MetricsClient



[simple] Turn 1: Book a trip from LA to NYC, July 10 to July 17, 1 traveler. Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.
I'll help you book this trip! Let me start by finding flights from LA to NYC for your dates, then look for hotels in Midtown with a pool.
Tool #1: flight_booking_assistant


2026-04-20 00:11:37 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-04-20 00:11:38 - INFO - Created event: 0000001776669097818#daac2c98


I'd be happy to help you find flights from Los Angeles to New York City for July 10-17! However, I should clarify that I don't have access to real-time flight data or booking systems, so I can't provide current prices or make actual reservations.

Here's how you can find the best flights for your criteria:

**Recommended Search Strategy:**
- **Departure:** LAX to JFK/LGA/EWR
- **Dates:** July 10-17, 2024
- **Budget:** $1,800 for roundtrip
- **Preference:** Morning departures

**Best Booking Platforms:**
1. **Google Flights** - Great for comparing prices and flexible dates
2. **Kayak** - Good for setting price alerts
3. **Expedia/Priceline** - Often have package deals
4. **Airline websites directly** (American, Delta, JetBlue, United)

**Morning Flight Options to Look For:**
- Departures between 6:00 AM - 11:00 AM
- American Airlines, Delta, and JetBlue typically offer good morning schedules
- United often has early departures from LAX

**Money-Saving Tips:**
- Book Tuesday/Wednesday de

2026-04-20 00:11:42 - INFO - Created event: 0000001776669102553#b0b401f4


...

?
Tool #3: itinerary_assistant


2026-04-20 00:11:56 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-04-20 00:11:56 - INFO - [itinerary] Loaded 3 turns
2026-04-20 00:11:56 - INFO - Created event: 0000001776669116302#18e88ad3


# 7-Day NYC Travel Itinerary
## July 10-17, 2024 | Solo Traveler

---

## **FLIGHT & ACCOMMODATION SUMMARY**

**✈️ FLIGHTS:**
- **Outbound:** July 10 - LAX to JFK/LGA (Morning departure 7:00-9:00 AM)
- **Return:** July 17 - JFK/LGA to LAX (Morning departure 8:00-10:00 AM)
- **Estimated Cost:** $600-800 roundtrip (within your $1,800 budget)

**🏨 ACCOMMODATION:**
**Le Parker Meridien** - West 57th Street
- **Dates:** July 10-17 (7 nights)
- **Pool:** Famous 42nd floor rooftop pool with city views
- **Estimated Rate:** $350-550/night = $2,450-3,850 total
- **Location:** Near Carnegie Hall and Central Park

---

## **DAY-BY-DAY ITINERARY**

### **DAY 1 - JULY 10 (ARRIVAL)**
**Morning/Afternoon:**
- 🛬 Arrive JFK/LGA (likely 3:00-5:00 PM EST after morning LAX departure)
- 🚖 Transit to Le Parker Meridien hotel (45-60 minutes)
- 🏨 Check-in and settle into room

**Evening:**
- 🍽️ Welcome dinner in Midtown (walking distance from hotel)
- 🌃 Times Square evening stroll (10-minute walk from hotel)


2026-04-20 00:12:12 - INFO - Created event: 0000001776669131990#fcbd6171


Perfect! I've found great options for your LA to NYC trip. Here's what I've put together:

**✈️ FLIGHTS**: Morning flights from LAX to NYC (July 10) and return (July 17) should cost $600-800, well within your $1,800 budget.

**🏨 HOTEL RECOMMENDATION**: **Le Parker Meridien** in Midtown - features an amazing 42nd-floor rooftop pool with city views! Located on West 57th Street near Carnegie Hall and Central Park.

**📅 COMPLETE ITINERARY**: I've created a full 7-day itinerary that includes major NYC attractions, Broadway shows, museums, and plenty of pool time to relax.

The hotel will be your biggest expense ($350-550/night), but you'll have budget left over for activities and dining. The location is perfect - walking distance to Times Square, Central Park, and Broadway theaters.

Would you like me to provide more details about any specific part of the trip, or do you have questions about booking any of these options?
Hub: Perfect! I've found great options for your LA to NYC trip. Here's

### Simple Session — Context Flow Trace

This shows exactly what each agent received (handoff + memory read) and produced (response) at every turn. Follow the flow to build a mental model of how context moves through the system.

In [7]:
display(Markdown(simple_conv_metrics.trace_report()))

### Context Flow Trace

════════════════════════════════════════════════════════════
 TURN 1
════════════════════════════════════════════════════════════

 **User message:** Book a trip from LA to NYC, July 10 to July 17, 1 traveler. Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.

---
**flight**

HANDOFF: Find flights from LA to NYC July 10 to July 17, 1 traveler, budget $1800, prefer morning flights

READ FROM MEMORY: _(empty)_

RESPONSE: I'd be happy to help you find flights from Los Angeles to New York City for July 10-17! However, I should clarify that I don't have access to real-time flight data or booking systems, so I can't provide current prices or make actual reservations.

Here's how you can find the best flights for your cr...

WRITTEN TO MEMORY: handoff + response above

---
**hotel**

HANDOFF: Find hotels in Midtown Manhattan NYC with pool, July 10-17 check-in dates, 1 guest

READ FROM MEMORY: Assistant: [flight] I'd be happy to help you find flights from Los Angeles to New York City for July 10-17! However, I should clarify that I don't have access to real-time flight data or booking systems, so I can't provide current prices or make actual reservations.

Here's how you can find the best...

RESPONSE: I'd be happy to help you find hotels in Midtown Manhattan with pools for your July 10-17 stay! Here are some excellent options with pools in that area:

**Luxury Options with Pools:**

**1. The Park Lane Hotel** - Central Park South
- **Pool:** Indoor rooftop pool with city views
- **Estimated Rate:...

WRITTEN TO MEMORY: handoff + response above

---
**itinerary**

HANDOFF: Build a travel itinerary for LA to NYC trip July 10-17, 1 traveler with morning flights and Midtown hotel with pool, budget $1800 for flights

READ FROM MEMORY: Assistant: [hotel] I'd be happy to help you find hotels in Midtown Manhattan with pools for your July 10-17 stay! Here are some excellent options with pools in that area:

**Luxury Options with Pools:**

**1. The Park Lane Hotel** - Central Park South
- **Pool:** Indoor rooftop pool with city views
...

RESPONSE: # 7-Day NYC Travel Itinerary
## July 10-17, 2024 | Solo Traveler

---

## **FLIGHT & ACCOMMODATION SUMMARY**

**✈️ FLIGHTS:**
- **Outbound:** July 10 - LAX to JFK/LGA (Morning departure 7:00-9:00 AM)
- **Return:** July 17 - JFK/LGA to LAX (Morning departure 8:00-10:00 AM)
- **Estimated Cost:** $600-...

WRITTEN TO MEMORY: handoff + response above

════════════════════════════════════════════════════════════
 TURN 2
════════════════════════════════════════════════════════════

 **User message:** Now build me a day-by-day itinerary for the trip based on the flight and hotel you found.


## Step 6 — Session with Feedback

Same initial request, but the user changes the budget mid-session. This tests:
- Does the coordinator re-dispatch with the updated budget?
- Do the spokes read the latest context from memory?
- Does the itinerary agent produce a plan consistent with the new budget?

In [8]:
feedback_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    "Actually, my budget changed to $1400 total for flights. "
    "Please redo the flight and hotel search with this new budget.",
    "Now build me a day-by-day itinerary for the trip based on the updated flight and hotel.",
]

feedback_metrics = run_multiagent_session(feedback_conversation, "feedback")

2026-04-20 00:12:22 - INFO - Found credentials in shared credentials file: ~/.aws/credentials



[feedback] Turn 1: Book a trip from LA to NYC, July 10 to July 17, 1 traveler. Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.
I'll help you book this trip from LA to NYC. Let me start by finding flights for you, then hotels, and finally put together your itinerary.
Tool #1: flight_booking_assistant


2026-04-20 00:12:25 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-04-20 00:12:26 - INFO - Created event: 0000001776669145829#6bd1b95a


I'd be happy to help you find flights from Los Angeles to New York City! However, I should clarify that I can't actually make real bookings or access live flight data. But I can guide you through the process and suggest what to look for.

**For your search criteria:**
- Route: LAX to NYC (JFK/LGA/EWR)
- Dates: July 10-17, 2024
- Passengers: 1 adult
- Budget: $1,800
- Preference: Morning departures

**Here's what I recommend:**

**Best booking sites to check:**
- Google Flights
- Expedia
- Kayak
- Direct airline websites (Delta, JetBlue, American, United)

**Morning flight options to look for:**
- **American Airlines**: LAX departures around 7:00-9:00 AM
- **JetBlue**: LAX to JFK, typically 8:00-10:00 AM departures
- **Delta**: LAX to JFK/LGA morning flights
- **United**: LAX to EWR morning options

**Expected pricing for July:**
- Economy: $400-800 roundtrip
- Premium Economy: $600-1,200
- Business: $1,200-2,000+

With your $1,800 budget, you could potentially book premium economy or e

2026-04-20 00:12:30 - INFO - Created event: 0000001776669150730#f2da6612


...

Based on your $1400 total budget, here's the reality:

**Updated Flight Budget:** $400-500 (economy, some morning options available)

**Remaining Hotel Budget:** $900-1000 for 7 nights

**Best Options with Pools:**
1. **Pod Brooklyn** - $770-980 total, rooftop pool, 15 mins to Manhattan
2. **Paper Factory Hotel (Queens)** - $700-910 total, rooftop pool, great Manhattan views
3. **YMCA West Side** - $840-1050 total, indoor pool, Times Square area

**The trade-off:** You'll need to stay outside Manhattan or choose basic accommodations to get a pool within budget.

**Quick questions:**
1. Are you flexible staying in Brooklyn/Queens with subway access to Manhattan?
2. Would you consider dropping the pool requirement for a Midtown location within budget?

The Pod Brooklyn offers the best compromise - rooftop pool, trendy area, and easy Manhattan access within your budget.
Hub: Based on your $1400 total budget, here's the reality:

**Updated Flight Budget:** $400-500 (economy, some morning o

2026-04-20 00:13:23 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-04-20 00:13:24 - INFO - [itinerary] Loaded 5 turns
2026-04-20 00:13:24 - INFO - Created event: 0000001776669204085#2be296e0


[itinerary] # New York City Travel Itinerary
**July 10-17, 2024 | Solo Traveler**

## Flight & Accommodation Summary
**Outbound Flight:** July 10
- LAX departure: Budget airline (Spirit/Frontier)
- Cost: ~$400-500 roundtrip
- NYC arrival: Afternoon/evening

**Return Flight:** July 17
- NYC departure to LAX
- Total flight cost: $400-500

**Hotel:** Pod Brooklyn (Williamsburg)
- Rooftop pool with Manhattan skyline views
- 15-minute subway ride to Manhattan (L train)
- July 10-17: ~$110-140/night
- 7 nights total: $770-980
- **Remaining budget for activities/food:** $420-630 ($60-90/day)

---

## Day-by-Day Itinerary

### **Day 1 - Wednesday, July 10: Arrival & Brooklyn Exploration**
- **Evening:** Land at JFK/LGA, take subway to Brooklyn
- **Check-in:** Pod Brooklyn, Williamsburg
- **8:00 PM:** Rooftop pool session with Manhattan views
- **9:00 PM:** Dinner at local Williamsburg spot ($25-35)
- **Budget:** $30-40

### **Day 2 - Thursday, July 11: Manhattan Introduction**
- **9:00 AM:** B

2026-04-20 00:13:45 - INFO - Created event: 0000001776669225749#20f1c9c3


Perfect! Here's your complete 7-day NYC itinerary with your $1400 budget:

**Your Budget Breakdown:**
- Flights: $400-500 
- Pod Brooklyn Hotel: $770-980 (7 nights)
- Daily expenses: $60-90/day for food & activities
- 7-day MetroCard: $33

**Key Highlights:**
- **Daily rooftop pool access** with Manhattan skyline views
- **15-minute subway rides** to Manhattan (L train from Bedford Ave)
- **Mix of free and paid attractions** to stay on budget
- **Brooklyn base** gives you local neighborhood experience

**Smart Budget Features:**
- Many FREE activities (Brooklyn Bridge, Central Park, High Line, Staten Island Ferry)
- One paid attraction (Top of the Rock OR Empire State Building)
- Affordable Williamsburg dining scene
- Pool time built into every day

**Daily Pool Schedule:**
- Afternoons: After Manhattan sightseeing
- Evenings: Sunset sessions with city views
- Less crowded times: Weekday mornings

The itinerary keeps you exactly on your $1400 budget while maximizing experiences and ens

### Feedback Session — Context Flow Trace

In [9]:

display(Markdown(feedback_metrics.trace_report()))

### Context Flow Trace

════════════════════════════════════════════════════════════
 TURN 1
════════════════════════════════════════════════════════════

 **User message:** Book a trip from LA to NYC, July 10 to July 17, 1 traveler. Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.

---
**flight**

HANDOFF: Book flights from Los Angeles to New York City, July 10 to July 17, 1 traveler, budget $1800, prefer morning flights

READ FROM MEMORY: _(empty)_

RESPONSE: I'd be happy to help you find flights from Los Angeles to New York City! However, I should clarify that I can't actually make real bookings or access live flight data. But I can guide you through the process and suggest what to look for.

**For your search criteria:**
- Route: LAX to NYC (JFK/LGA/EW...

WRITTEN TO MEMORY: handoff + response above

---
**hotel**

HANDOFF: Book hotel in Midtown Manhattan NYC, July 10-17, 1 traveler, must have a pool

READ FROM MEMORY: Assistant: [flight] I'd be happy to help you find flights from Los Angeles to New York City! However, I should clarify that I can't actually make real bookings or access live flight data. But I can guide you through the process and suggest what to look for.

**For your search criteria:**
- Route: LA...

RESPONSE: I'd be happy to help you find a hotel in Midtown Manhattan with a pool for your July 10-17 stay! Since you're looking for accommodations that complement your flight dates, let me suggest some excellent options.

**Top Midtown Hotels with Pools:**

**1. Le Parker Meridien New York**
- Location: 57th ...

WRITTEN TO MEMORY: handoff + response above

---
**itinerary**

HANDOFF: Create travel itinerary for LA to NYC trip July 10-17, 1 traveler, morning flights with $1800 budget, Midtown hotel with pool, 7-day stay

READ FROM MEMORY: Assistant: [hotel] I'd be happy to help you find a hotel in Midtown Manhattan with a pool for your July 10-17 stay! Since you're looking for accommodations that complement your flight dates, let me suggest some excellent options.

**Top Midtown Hotels with Pools:**

**1. Le Parker Meridien New York*...

RESPONSE: # New York City Travel Itinerary
**July 10-17, 2024 | Solo Traveler**

## Flight & Accommodation Summary
**Outbound Flight:** July 10
- LAX departure: 8:00 AM (recommended JetBlue or American)
- NYC arrival: ~4:30 PM EST
- Budget allocation: ~$400-600

**Return Flight:** July 17  
- NYC departure: 9...

WRITTEN TO MEMORY: handoff + response above

════════════════════════════════════════════════════════════
 TURN 2
════════════════════════════════════════════════════════════

 **User message:** Actually, my budget changed to $1400 total for flights. Please redo the flight and hotel search with this new budget.

---
**flight**

HANDOFF: Book flights from Los Angeles to New York City, July 10 to July 17, 1 traveler, total budget $1400 for entire trip including flights and hotel, prefer morning flights

READ FROM MEMORY: Assistant: [itinerary] # New York City Travel Itinerary
**July 10-17, 2024 | Solo Traveler**

## Flight & Accommodation Summary
**Outbound Flight:** July 10
- LAX departure: 8:00 AM (recommended JetBlue or American)
- NYC arrival: ~4:30 PM EST
- Budget allocation: ~$400-600

**Return Flight:** July ...

RESPONSE: I understand you'd like help with booking, but I should clarify that I can't make actual reservations. However, I can help you plan within your $1,400 total budget for both flights and hotel.

**Budget Reality Check:**
With $1,400 total for flights + hotel for 7 nights in NYC during peak summer seas...

WRITTEN TO MEMORY: handoff + response above

---
**hotel**

HANDOFF: Budget hotel options in NYC or nearby areas with pool, July 10-17, total remaining budget around $700-1000 after flights, need 7 nights

READ FROM MEMORY: Assistant: [flight] I understand you'd like help with booking, but I should clarify that I can't make actual reservations. However, I can help you plan within your $1,400 total budget for both flights and hotel.

**Budget Reality Check:**
With $1,400 total for flights + hotel for 7 nights in NYC dur...

RESPONSE: [hotel] I understand you have $700-1000 remaining for 7 nights after flights - that's a tight but workable budget! Here are realistic hotel options with pools in that range:

**NYC Area Hotels with Pools ($700-1000 total):**

**1. YMCA Hotels (Manhattan)**
- **West Side YMCA**: 47th & 8th Avenue
- P...

WRITTEN TO MEMORY: handoff + response above

════════════════════════════════════════════════════════════
 TURN 3
════════════════════════════════════════════════════════════

 **User message:** Now build me a day-by-day itinerary for the trip based on the updated flight and hotel.

---
**itinerary**

HANDOFF: Create travel itinerary for LA to NYC trip July 10-17, 1 traveler, budget flights $400-500, staying at Pod Brooklyn with rooftop pool, total budget $1400, 7-day stay with subway access to Manhattan

READ FROM MEMORY: Assistant: [hotel] [hotel] I understand you have $700-1000 remaining for 7 nights after flights - that's a tight but workable budget! Here are realistic hotel options with pools in that range:

**NYC Area Hotels with Pools ($700-1000 total):**

**1. YMCA Hotels (Manhattan)**
- **West Side YMCA**: 47...

RESPONSE: [itinerary] # New York City Travel Itinerary
**July 10-17, 2024 | Solo Traveler**

## Flight & Accommodation Summary
**Outbound Flight:** July 10
- LAX departure: Budget airline (Spirit/Frontier)
- Cost: ~$400-500 roundtrip
- NYC arrival: Afternoon/evening

**Return Flight:** July 17
- NYC departure...

WRITTEN TO MEMORY: handoff + response above


## Step 7 — Run LLM-as-Judge Evaluations

This runs Claude Opus as a judge on every agent call in both sessions. It evaluates context freshness, handoff completeness, utilization, write accuracy, redundancy, and cross-agent consistency.

In [10]:
print("Evaluating simple session...")
simple_conv_metrics.evaluate_all()

print("Evaluating feedback session...")
feedback_metrics.evaluate_all()

2026-04-20 00:13:50 - INFO - Running LLM-as-judge evaluations...


Evaluating simple session...


2026-04-20 00:15:12 - INFO - LLM-as-judge evaluations complete.
2026-04-20 00:15:12 - INFO - Running LLM-as-judge evaluations...


Evaluating feedback session...


2026-04-20 00:18:28 - INFO - LLM-as-judge evaluations complete.


## Step 8 — Results: Simple Session

In [11]:
display(Markdown("## Simple Session\n\n" + simple_conv_metrics.context_metrics_report()))
display(Markdown(simple_conv_metrics.latency_metrics_report()))

## Simple Session

### Memory Context Metrics

| Turn | Agent | CCR | Freshness | Handoff Complete | Context Util. | Write Accuracy | Redundancy |
|------|-------|-----|-----------|------------------|---------------|----------------|------------|
| 1 | flight | 0.67 | 5/5 | 5/5 | 5/5 | 5/5 | 5/5 |
| 1 | hotel | 0.57 | 3/5 | 4/5 | 1/5 | 2/5 | 1/5 |
| 1 | itinerary | 0.98 | 5/5 | 5/5 | 2/5 | 3/5 | 2/5 |

**State Consistency (per turn):**
- Turn 1: 4/5 — All agents consistently reference the same travel dates (July 10-17), locations (LAX to NYC), budget ($1,800 for flights), morning departure preference, and hotel recommendations with pools in Midtown Manhattan, with only minor variations in presentation style.
- Turn 2: 5/5 — Single agent, no cross-check needed

<details><summary>LLM Judge Reasoning</summary>

**Turn 1 — flight:**
- context_freshness: No prior context (first call)
- handoff_completeness: The query includes all flight-relevant information: origin/destination cities, dates, traveler count, budget, and flight time preference.
- context_utilization: No context to utilize (first call)
- write_accuracy: The response accurately addresses all input requirements (LA to NYC, July 10-17, 1 traveler, $1800 budget, morning preference) while appropriately disclaiming lack of real-time data access and providing factually correct general travel advice.
- redundant_context: No context retrieved

**Turn 1 — hotel:**
- context_freshness: The context reflects the flight requirements accurately but completely misses the new hotel requirement for Midtown with a pool.
  - Stale fields: ['hotel_location', 'hotel_amenities']
- handoff_completeness: The query includes all essential information for hotel booking except the budget constraint which could be relevant for hotel selection.
  - Missing: ['budget information']
- context_utilization: The agent completely ignored the flight search context and instead provided hotel recommendations with pools, which was not requested by the user.
- write_accuracy: The response contains multiple fabricated details including specific hotel names, addresses, pool descriptions, and price ranges that cannot be verified as accurate without real-time data access.
- redundant_context: The context is entirely about flight booking while the agent is a hotel agent, making 100% of the content irrelevant to the hotel agent's task.

**Turn 1 — itinerary:**
- context_freshness: The context fully reflects all the latest user requirements including LA to NYC travel, July 10-17 dates, 1 traveler, $1800 flight budget, morning flight preference, and Midtown hotel with pool.
- handoff_completeness: The handoff includes all essential facts from the original request: dates (July 10-17), locations (LA to NYC), traveler count (1), budget ($1800 for flights), and preferences (morning flights, Midtown hotel with pool).
- context_utilization: The agent primarily used only one hotel recommendation (Le Parker Meridien) from the retrieved context while ignoring the other hotel options, pool access fees, booking recommendations, and important notes about seasonal hours and weekend rates.
- write_accuracy: The response accurately addresses the travel dates, solo traveler, morning flights, and Midtown hotel with pool requirements, but fabricates specific hotel details (Le Parker Meridien's rates exceed the $1800 budget for flights alone) and includes overly specific information not derivable from the input.
- redundant_context: The context contains a complete hotel response that is entirely irrelevant to the flight query, making up approximately 60% of the total content, while the flight response itself is appropriate.

</details>

### Memory Latency Metrics

| Turn | Agent | Read (s) | Write (s) | Reasoning (s) | Coord Lat % | Coord Tok % |
|------|-------|----------|-----------|---------------|-------------|-------------|
| 1 | flight | 0.20 | 0.40 | 4.34 | 12% | 0% |
| 1 | hotel | 0.11 | 0.37 | 7.75 | 6% | 100% |
| 1 | itinerary | 0.12 | 0.41 | 15.36 | 3% | 100% |

**Avg Coordination Latency %:** 6%
**Avg Coordination Token %:** 100%

## Step 9 — Results: Session with Feedback

In [12]:
display(Markdown("## Session with Feedback\n\n" + feedback_metrics.context_metrics_report()))
display(Markdown(feedback_metrics.latency_metrics_report()))

## Session with Feedback

### Memory Context Metrics

| Turn | Agent | CCR | Freshness | Handoff Complete | Context Util. | Write Accuracy | Redundancy |
|------|-------|-----|-----------|------------------|---------------|----------------|------------|
| 1 | flight | 0.81 | 5/5 | 5/5 | 5/5 | 4/5 | 5/5 |
| 1 | hotel | 0.53 | 5/5 | 4/5 | 1/5 | 2/5 | 1/5 |
| 1 | itinerary | 0.95 | 5/5 | 5/5 | 5/5 | 2/5 | 2/5 |
| 2 | flight | 1.42 | 1/5 | 2/5 | 2/5 | 2/5 | 2/5 |
| 2 | hotel | 1.15 | 1/5 | 4/5 | 4/5 | 2/5 | 2/5 |
| 3 | itinerary | 2.26 | 1/5 | 4/5 | 4/5 | 4/5 | 2/5 |

**State Consistency (per turn):**
- Turn 1: 4/5 — The agents maintain consistency on dates (July 10-17), route (LAX to NYC), budget ($1,800 for flights), and hotel recommendation (Le Parker Meridien at $450-650/night), with only minor variations in how they present the information.
- Turn 2: 4/5 — Both agents agree on the core facts (budget of $1400 total, $700-1100 remaining after flights for 7 nights) with only minor differences in how they present the feasibility of finding hotels with pools in that price range.
- Turn 3: 5/5 — Single agent, no cross-check needed

<details><summary>LLM Judge Reasoning</summary>

**Turn 1 — flight:**
- context_freshness: No prior context (first call)
- handoff_completeness: The query includes all flight-relevant information: origin/destination cities, dates, traveler count, budget, and flight time preference.
- context_utilization: No context to utilize (first call)
- write_accuracy: The response accurately reflects the input parameters and provides reasonable general guidance, but assumes 2024 as the year when no year was specified in the input.
- redundant_context: No context retrieved

**Turn 1 — hotel:**
- context_freshness: The context fully reflects all the latest user requirements including LA to NYC route, July 10-17 dates, 1 traveler, $1800 budget, and morning flight preference.
- handoff_completeness: The handoff includes all essential facts (location, dates, guest count, pool requirement) but omits the budget constraint which could be relevant for hotel selection.
  - Missing: ['budget information']
- context_utilization: The agent completely ignored the flight booking context and instead responded about hotels with pools, which was not requested by the user.
- write_accuracy: The response contains multiple fabricated details including specific hotel rates, exact addresses, and assumes flight dates that were never mentioned in the input, making it significantly inaccurate.
- redundant_context: This flight booking context is entirely irrelevant for a hotel agent who needs hotel-related information, not flight details.

**Turn 1 — itinerary:**
- context_freshness: The context fully reflects all the latest user requirements including LA to NYC route, July 10-17 dates, 1 traveler, $1800 flight budget, morning flight preference, and Midtown hotel with pool.
- handoff_completeness: The handoff includes all essential facts: origin/destination cities, specific dates, traveler count, flight budget, flight time preference, hotel location preference, and hotel amenity requirement.
- context_utilization: The agent fully incorporated all relevant details from both contexts including specific flight times, airlines, prices, hotel recommendations with pool amenities, and booking site suggestions into a comprehensive itinerary.
- write_accuracy: The agent's hotel budget calculation ($3,150-4,550 for 7 nights) far exceeds the stated $1,800 total budget, making the itinerary financially impossible as presented.
- redundant_context: The context contains two separate conversations about hotels and flights that appear unrelated, with the hotel information being completely irrelevant to the flight booking request.

**Turn 2 — flight:**
- context_freshness: The context shows a flight budget of $800-1,200 out of $1,800 total, while the user just changed their budget to $1,400 total for flights.
  - Stale fields: ['flight_budget', 'total_budget']
- handoff_completeness: The query incorrectly states the entire $1400 budget is for the trip when the user specifically said $1400 is just for flights, and it asks to 'book' rather than 'search' for flights.
  - Missing: ['correct budget allocation ($1400 for flights only, not entire trip)', 'search request not booking request']
- context_utilization: The agent completely ignored the specific flight times, hotel name (Le Parker Meridien), and detailed pricing from the context, instead providing generic budget advice that contradicts the retrieved information.
- write_accuracy: The agent fabricated a requirement for 'Midtown hotels with pools' that was never mentioned in the input, and built much of their response around this non-existent constraint.
- redundant_context: The context contains extensive day-by-day itinerary details that are completely irrelevant to a flight agent's task, with only the flight summary section (approximately 15-20% of content) being useful.

**Turn 2 — hotel:**
- context_freshness: The context shows a hotel (Le Parker Meridien) costing $3,150-4,550 for 7 nights, which completely exceeds the user's $1,400 total budget for both flights and hotel.
  - Stale fields: ['hotel_selection', 'hotel_price', 'total_budget_allocation']
- handoff_completeness: The handoff includes most essential information (dates, location, budget range, pool requirement, duration) but omits the context that this is a revised search and doesn't specify the number of guests.
  - Missing: ['number of guests', 'context about budget revision']
- context_utilization: The agent effectively used the budget breakdown and hotel price ranges from the context, but didn't mention the suggested booking platforms (Hotels.com, Booking.com) or the alternative of hostels that were provided in the retrieved context.
- write_accuracy: The response contains significant inaccuracies including fabricated details about specific hotels having pools (Pod Brooklyn and Paper Factory Hotel do not have rooftop pools, and some YMCAs listed may not have pools) and potentially inaccurate pricing for peak July dates in NYC.
- redundant_context: The context contains extensive flight booking details and budget discussions that are irrelevant to a hotel agent, plus the itinerary section is incomplete and mostly focuses on daily activities rather than hotel-specific information.

**Turn 3 — itinerary:**
- context_freshness: The context contains flight and hotel search information but completely lacks any day-by-day itinerary information that the user is now requesting.
  - Stale fields: ['itinerary', 'daily_activities', 'day_by_day_plan']
- handoff_completeness: The handoff includes most essential information but omits the specific flight details and exact hotel booking dates that would be needed to build a precise day-by-day itinerary.
  - Missing: ['specific flight times/dates', 'exact hotel check-in/check-out dates', 'any activities or interests preferences']
- context_utilization: The agent incorporated most of the retrieved hotel context by selecting Pod Brooklyn as the main recommendation and accurately using its details (location, pool, price range), but omitted the other hotel options that were provided in the context.
- write_accuracy: The response accurately addresses all input requirements (dates, budget, Pod Brooklyn with rooftop pool, subway access) but makes minor assumptions about specific subway lines and exact pricing that weren't provided in the input.
- redundant_context: The context contains two complete responses about hotels when the user is asking about flights, making most of the hotel information irrelevant to the flight booking request.

</details>

### Memory Latency Metrics

| Turn | Agent | Read (s) | Write (s) | Reasoning (s) | Coord Lat % | Coord Tok % |
|------|-------|----------|-----------|---------------|-------------|-------------|
| 1 | flight | 0.08 | 0.40 | 4.62 | 10% | 0% |
| 1 | hotel | 0.12 | 0.41 | 4.84 | 10% | 100% |
| 1 | itinerary | 0.11 | 0.38 | 12.23 | 4% | 100% |
| 2 | flight | 0.14 | 0.44 | 5.50 | 10% | 100% |
| 2 | hotel | 0.13 | 0.44 | 7.76 | 7% | 100% |
| 3 | itinerary | 0.16 | 0.45 | 21.27 | 3% | 100% |

**Avg Coordination Latency %:** 5%
**Avg Coordination Token %:** 100%

## Step 10 — Side-by-Side Comparison

In [13]:
display(Markdown(MetricsCollector.comparison_report(
    simple_conv_metrics, feedback_metrics,
    label_a="Simple Session", label_b="Session with Feedback")))

### Simple Session vs Session with Feedback

| Metric | Simple Session | Session with Feedback | Delta |
|--------|----------|----------|-------|
| Avg Context Freshness | 4.3/5 | 3.0/5 | -1.3 |
| Avg Handoff Completeness | 4.7/5 | 4.0/5 | -0.7 |
| Avg Context Utilization | 2.7/5 | 3.5/5 | +0.8 |
| Avg Write Accuracy | 3.3/5 | 2.7/5 | -0.7 |
| Avg Context Efficiency | 2.7/5 | 2.3/5 | -0.3 |
| Avg State Consistency | 4.5/5 | 4.3/5 | -0.2 |

## Clean Up

In [14]:
# Uncomment to delete the memory resource
# memory_client.delete_memory_and_wait(memory_id=memory_id, max_wait=300, poll_interval=10)